In [ ]:
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import KFold
from multimodal_dataset import MultimodalDataset

df = pd.read_csv(r"/mnt/c/Users/Kunny/Research/Project/BiConVarNet/filtered_variants_cleaned_final.tsv", sep="\t", )

# 10-fold 
kf = KFold(n_splits=10, shuffle=True, random_state=42)
splits = list(kf.split(df))
train_idx, val_idx = splits[0]

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
        
voxel_cache_dir = "/mnt/e/CAGI_data/voxel_cache_4_noRSA"
msa_dict_path = "/mnt/e/CAGI_data/msa_dict_valid_new.pkl"

train_dataset = MultimodalDataset(train_df, voxel_cache_dir, msa_dict_path, voxel_aug=True, msa_aug=True)
val_dataset   = MultimodalDataset(val_df, voxel_cache_dir, msa_dict_path)

train_loader = DataLoader(train_dataset, batch_size=88, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

In [ ]:
# 전체 분포
print("=== 전체 데이터 분포 ===")
print(df["Label"].value_counts(normalize=True))  
print(df["Label"].value_counts())                

# train 분포
print("\n=== Train 데이터 분포 ===")
print(train_df["Label"].value_counts(normalize=True))
print(train_df["Label"].value_counts())

# val 분포
print("\n=== Validation 데이터 분포 ===")
print(val_df["Label"].value_counts(normalize=True))
print(val_df["Label"].value_counts())

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score
from tqdm import tqdm

# --- Configuration & Hyperparameters ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 3e-3
T_MAX = 100
NUM_EPOCHS = 100
MIX_ALPHA = 0.4
MIX_LOSS_WEIGHT = 0.6
SAVE_PATH = "/mnt/e/CAGI_data/best_model_clip.pth"

In [ ]:
# --- Utility Functions ---

def contrastive_loss(logits: torch.Tensor) -> torch.Tensor:
    """Standard InfoNCE loss for contrastive learning."""
    return F.cross_entropy(logits, torch.arange(len(logits), device=logits.device))

def compute_clip_loss(similarity: torch.Tensor) -> torch.Tensor:
    """Symmetric CLIP loss (Voxel-to-MSA and MSA-to-Voxel)."""
    return (contrastive_loss(similarity) + contrastive_loss(similarity.t())) / 2

def fusemix(voxel_feat, msa_feat, alpha=MIX_ALPHA):
    """
    Applies Mixup in the multimodal embedding space.
    Blends two different samples to regularize the feature space.
    """
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(voxel_feat.size(0), device=voxel_feat.device)

    voxel_mix = lam * voxel_feat + (1 - lam) * voxel_feat[idx]
    msa_mix = lam * msa_feat + (1 - lam) * msa_feat[idx]

    return voxel_mix, msa_mix

In [ ]:
# --- Training and Evaluation Functions ---

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="[Train]"):
        # Move batch to device
        voxel = batch["voxel"].to(DEVICE)
        ref_idx = batch["ref_idx"].to(DEVICE)
        mut_idx = batch["mut_idx"].to(DEVICE)
        msa = batch["msa"].to(DEVICE)
        label = batch["label"].float().to(DEVICE)

        optimizer.zero_grad()

        # 1. Forward Pass
        out = model(voxel, ref_idx, mut_idx, msa)
        
        # 2. Main Classification Loss (BCE)
        logits = out["logits"].squeeze(-1)
        cls_loss = criterion(logits, label)

        # 3. Original CLIP Loss
        clip_loss_val = compute_clip_loss(out["logits_per_voxel"])

        # 4. FuseMix (Feature Augmentation) Loss
        v_mix, m_mix = fusemix(out["voxel_feat"], out["msa_feat"])
        v_mix_norm = F.normalize(v_mix, dim=-1)
        m_mix_norm = F.normalize(m_mix, dim=-1)
        
        # Recalculate similarity for mixed features
        sim_mix = torch.matmul(v_mix_norm, m_mix_norm.T) * model.logit_scale.exp()
        loss_mix = compute_clip_loss(sim_mix)

        # 5. Total Loss & Optimization
        batch_loss = cls_loss + clip_loss_val + (MIX_LOSS_WEIGHT * loss_mix)
        batch_loss.backward()
        optimizer.step()

        total_loss += batch_loss.item() * voxel.size(0)

    return total_loss / len(loader.dataset)

def validate(model, loader, criterion):
    model.eval()
    val_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="[Val]"):
            voxel = batch["voxel"].to(DEVICE)
            ref_idx = batch["ref_idx"].to(DEVICE)
            mut_idx = batch["mut_idx"].to(DEVICE)
            msa = batch["msa"].to(DEVICE)
            label = batch["label"].float().to(DEVICE)

            out = model(voxel, ref_idx, mut_idx, msa)
            logits = out["logits"].squeeze(-1)
            
            loss = criterion(logits, label)
            probs = torch.sigmoid(logits)

            val_loss += loss.item() * voxel.size(0)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    # Calculate Metrics
    metrics = {
        "loss": val_loss / len(loader.dataset),
        "pr_auc": average_precision_score(all_labels, all_probs),  
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "acc": accuracy_score(all_labels, [1 if p >= 0.5 else 0 for p in all_probs])
    }
    return metrics

In [ ]:
from Models import EvoStructCLIP

# --- Main Execution ---

# Model, Criterion, Optimizer, Scheduler
model = EvoStructCLIP(voxel_ch=46, mb_layers=6, embed_dim=128, use_concat=True).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

best_pr_auc = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_metrics = validate(model, val_loader, criterion)
    scheduler.step()

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f}")
    print(f"PR-AUC: {val_metrics['pr_auc']:.4f} | ROC-AUC: {val_metrics['roc_auc']:.4f} | Acc: {val_metrics['acc']:.4f}")

    # Save best model based on PR-AUC
    if val_metrics['pr_auc'] > best_pr_auc:
        best_pr_auc = val_metrics['pr_auc']
        torch.save(model.state_dict(), SAVE_PATH)
        print(f">>> Best model saved with PR-AUC: {best_pr_auc:.4f}")